In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from numpy.fft import fft, ifft
from numpy.random import default_rng
from sklearn.utils import check_random_state  # Fix for random state handling
import pandas as pd
from scipy import stats
import seaborn as sns
import os
from collections import defaultdict


Functions and code to perform permutation testing for each arousal time course

In [ ]:
def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5 * (np.log1p(r) - np.log1p(-r))

def fisher_z_inv(z):
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)

def compute_observed_value(predicted_arousal, observed_arousal, stat="median", corr_type="pearson", return_all_r=False):
    if predicted_arousal.shape[0] != observed_arousal.shape[0]:
        min_subjects = min(predicted_arousal.shape[0], observed_arousal.shape[0])
        print(f"Warning: Mismatched subject counts! Using first {min_subjects} subjects.")
        predicted_arousal = predicted_arousal[:min_subjects]
        observed_arousal = observed_arousal[:min_subjects]

    n_subj = predicted_arousal.shape[0]
    observed_correlations = []

    for subj in range(n_subj):
        valid_mask = ~np.isnan(predicted_arousal[subj]) & ~np.isnan(observed_arousal[subj])
        if np.sum(valid_mask) > 1:
            if corr_type == "pearson":
                corr, _ = pearsonr(predicted_arousal[subj][valid_mask], observed_arousal[subj][valid_mask])
            elif corr_type == "spearman":
                corr, _ = spearmanr(predicted_arousal[subj][valid_mask], observed_arousal[subj][valid_mask])
            else:
                raise ValueError("Invalid correlation type. Choose 'pearson' or 'spearman'.")
            observed_correlations.append(corr)

    if len(observed_correlations) == 0:
        raise ValueError("No valid correlations computed. Check your data for NaN values.")

    observed_correlations = np.array(observed_correlations)

    if return_all_r:
        return observed_correlations

    z_vals = fisher_z(observed_correlations)
    z_stat = np.median(z_vals) if stat == "median" else np.mean(z_vals)
    return fisher_z_inv(z_stat)

def phase_randomize(data, random_state=None):
    random_state = check_random_state(random_state)
    data = np.array(data)
    fft_data = fft(data, axis=0)
    if data.shape[0] % 2 == 0:
        pos_freq = np.arange(1, data.shape[0] // 2)
        neg_freq = np.arange(data.shape[0] - 1, data.shape[0] // 2, -1)
    else:
        pos_freq = np.arange(1, (data.shape[0] - 1) // 2 + 1)
        neg_freq = np.arange(data.shape[0] - 1, (data.shape[0] - 1) // 2, -1)
    if len(data.shape) == 1:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq)))
        fft_data[pos_freq] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq] *= np.exp(-1j * phase_shifts)
    else:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq), data.shape[1]))
        fft_data[pos_freq, :] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq, :] *= np.exp(-1j * phase_shifts)
    return np.real(ifft(fft_data, axis=0))


def permutation_test(predicted_arousal, observed_arousal, 
                     null_type="phase_randomization", stat="median", 
                     n_permutations=10000, save_path=None, corr_type="pearson"):
    
    # Compute and save observed r-values
    observed_rvals = compute_observed_value(predicted_arousal, observed_arousal, stat, corr_type, return_all_r=True)
    z_vals = fisher_z(observed_rvals)
    z_stat = np.median(z_vals) if stat == "median" else np.mean(z_vals)
    observed_value = fisher_z_inv(z_stat)

    print(f"Observed {stat} {corr_type} correlation: {observed_value:.4f}")

    # Generate null distribution
    null_distributions = []
    for _ in range(n_permutations):
        if null_type == "phase_randomization":
            shuffled_observed_arousal = np.array([phase_randomize(a) for a in observed_arousal])
        elif null_type == "between_subject":
            shuffled_indices = np.random.permutation(observed_arousal.shape[0])
            shuffled_observed_arousal = observed_arousal[shuffled_indices]
        else:
            raise ValueError("Invalid null_type. Choose 'phase_randomization' or 'between_subject'.")

        shuffled_value = compute_observed_value(predicted_arousal, shuffled_observed_arousal, stat, corr_type)
        null_distributions.append(shuffled_value)

    null_distributions = np.array(null_distributions)
    p_value = (np.sum(null_distributions >= observed_value) + 1) / (n_permutations + 1)
    print(f"P-value: {p_value:.4f}")

    # Save results
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        base = os.path.splitext(save_path)[0]

        # Save null distribution and observed r-values
        np.save(base + "_null_distribution.npy", null_distributions)
        np.save(base + "_observed_rvals.npy", observed_rvals)

        # Save p-value
        with open(base + "_pval.txt", "w") as f:
            f.write(f"{p_value:.6f}")

        # Save individual subject correlations
        df = pd.DataFrame({'subject': np.arange(len(observed_rvals)), f'{corr_type}_r': observed_rvals})
        df.to_csv(base + "_subject_rvals.csv", index=False)

        # Save group-level observed correlation
        with open(base + f"_{stat}_observed_r.txt", "w") as f:
            f.write(f"{observed_value:.6f}")

        print(f"Saved results to {base}_*.npy/.txt/.csv")

    # Optional plot
    plt.figure(figsize=(8,6))
    plt.hist(null_distributions, bins=50, color='gray', alpha=0.7, density=True, label=f"Null Distribution ({null_type})")
    plt.axvline(observed_value, color='red', linestyle='dotted', linewidth=2, label=f"Observed {stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.xlabel(f"{stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.ylabel("Density")
    plt.title(f"Permutation Test ({null_type.replace('_', ' ').title()})")
    plt.grid(True)
    plt.legend()
    plt.show()

    return observed_value, null_distributions, p_value


In [ ]:
# Shared base directory
base = "../4_results/2_classification/"

# Each entry is: (subfolder, predicted filename, observed filename)
file_entries = [
    ("2_EEG_Rest_within", "EEG_Rest_within_predicted.npy", "EEG_Rest_within_observed.npy"),
    ("3_Pupil_Rest_within", "Pupil_Rest_within_predicted.npy", "Pupil_Rest_within_observed.npy"),
    ("4_EEG_Rest_from_Pupil_Rest", "EEG_Rest_from_Pupil_Rest_predicted.npy", "EEG_Rest_from_Pupil_Rest_observed.npy"),
    ("5_Pupil_Rest_from_EEG_Rest", "Pupil_Rest_from_EEG_Rest_predicted.npy", "Pupil_Rest_from_EEG_Rest_observed.npy"),
    ("6_Pupil_Story_from_Pupil_Rest", "Pupil_Story_from_Pupil_Rest_predicted.npy", "Pupil_Story_from_Pupil_Rest_observed.npy"),
    ("7_Pupil_Rest_from_Pupil_Story", "Pupil_Rest_from_Pupil_Story_predicted.npy", "Pupil_Rest_from_Pupil_Story_observed.npy"),
    ("8_Pupil_Story_from_EEG_Rest", "Pupil_Story_from_EEG_Rest_predicted.npy", "Pupil_Story_from_EEG_Rest_observed.npy"),
    ("9_EEG_Rest_from_Pupil_Story", "EEG_Rest_from_Pupil_Story_predicted.npy", "EEG_Rest_from_Pupil_Story_observed.npy"),
    ("10_EEG_Rest_from_Movies", "EEG_Rest_from_Movies_predicted.npy", "EEG_Rest_from_Movies_observed.npy"),
    ("12_Pupil_Rest_from_Movies", "Pupil_Rest_from_Movies_predicted.npy", "Pupil_Rest_from_Movies_observed.npy"),
    ("14_Pupil_Story_from_Movies", "Pupil_Story_from_Movies_predicted.npy", "Pupil_Story_from_Movies_observed.npy"),

]


# Loop through each entry
for folder, pred_file, obs_file in file_entries:
    pred_path = os.path.join(base, folder, pred_file)
    obs_path = os.path.join(base, folder, obs_file)

    if not os.path.exists(pred_path) or not os.path.exists(obs_path):
        print(f"Skipping {folder}: Missing one or both files.")
        continue

    predicted_arousal = np.load(pred_path)
    observed_arousal = np.load(obs_path)

    save_path = os.path.join(base, folder, "model5_p05_mean")
    print(f"Running {folder}")

    observed_value, null_distribution, p_value = permutation_test(
        predicted_arousal,
        observed_arousal,
        null_type="phase_randomization",
        stat="mean",
        n_permutations=10000,
        corr_type="pearson",
        save_path=save_path
    )


Combine RVals for Predicting Movies

In [ ]:
FF_observed = np.load('../4_results/2_classification/1_Movies_within/FF_from_Sherlock_observed.npy')
Sher_observed = np.load('../4_results/2_classification/1_Movies_within/Sherlock_from_FF_observed.npy')

FF_Sher_predicted = np.load('../4_results/2_classification/1_Movies_within/Sherlock_from_FF_predicted.npy')
Sher_FF_predicted = np.load('../4_results/2_classification/1_Movies_within/FF_from_Sherlock_predicted.npy')

pupil_FF_predicted = np.load('../4_results/2_classification/13_Movies_from_Pupil_Rest/FilmFest_from_Pupil_Rest_predicted.npy')
pupil_Sher_predicted = np.load('../4_results/2_classification/13_Movies_from_Pupil_Rest/Sherlock_from_Pupil_Rest_predicted.npy')

EEG_FF_predicted = np.load('../4_results/2_classification/11_Movies_from_EEG_Rest/FilmFest_from_EEG_Rest_predicted.npy')
EEG_Sher_predicted = np.load('../4_results/2_classification/11_Movies_from_EEG_Rest/Sherlock_from_EEG_Rest_predicted.npy')

paranoia_FF_predicted = np.load('../4_results/2_classification/15_Movies_from_Pupil_Story/FilmFest_from_Pupil_Story_predicted.npy')
paranoia_Sher_predicted = np.load('../4_results/2_classification/15_Movies_from_Pupil_Story/Sherlock_from_Pupil_Story_predicted.npy')


In [ ]:
def permutation_test_toSR(toFF_predicted, FF_observed, toS_predicted, S_observed, 
                     null_type="phase_randomization", stat="median", 
                     n_permutations=10000, save_path=None, corr_type="pearson"):
    
    # Compute and save observed r-values
    all_r = []
    for i in range(toFF_predicted.shape[0]):
        all_r.append(pearsonr(toFF_predicted[i], FF_observed[i])[0])
    for i in range(toS_predicted.shape[0]):
        all_r.append(pearsonr(toS_predicted[i], S_observed[i])[0])
    observed_rvals = np.array(all_r)
    z_vals = fisher_z(observed_rvals)
    z_stat = np.median(z_vals) if stat == "median" else np.mean(z_vals)
    observed_value = fisher_z_inv(z_stat)

    print(f"Observed {stat} {corr_type} correlation: {observed_value:.4f}")

    # Generate null distribution
    null_distributions = []
    for _ in range(n_permutations):
        if null_type == "phase_randomization":
            shuffled_observed_arousal_FF = np.array([phase_randomize(a) for a in FF_observed])
            shuffled_observed_arousal_S = np.array([phase_randomize(a) for a in S_observed])
        else:
            raise ValueError("Invalid null_type. Choose 'phase_randomization'.")

        fake_r = []
        for i in range(shuffled_observed_arousal_FF.shape[0]):
            fake_r.append(pearsonr(toFF_predicted[i], shuffled_observed_arousal_FF[i])[0])
        for i in range(shuffled_observed_arousal_S.shape[0]):
            fake_r.append(pearsonr(toS_predicted[i], shuffled_observed_arousal_S[i])[0])
        fake_r = np.array(fake_r)
        fake_z_vals = fisher_z(np.array(fake_r))
        fake_z_stat = np.median(fake_z_vals) if stat == "median" else np.mean(fake_z_vals)
        null_distributions.append(fisher_z_inv(fake_z_stat))


    null_distributions = np.array(null_distributions)
    p_value = (np.sum(null_distributions >= observed_value) + 1) / (n_permutations + 1)
    print(f"P-value: {p_value:.4f}")

    # Save results
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        base = os.path.splitext(save_path)[0]

        # Save null distribution and observed r-values
        np.save(base + "_null_distribution.npy", null_distributions)
        np.save(base + "_observed_rvals.npy", observed_rvals)

        # Save p-value
        with open(base + "_pval.txt", "w") as f:
            f.write(f"{p_value:.6f}")

        # Save individual subject correlations
        df = pd.DataFrame({'subject': np.arange(len(observed_rvals)), f'{corr_type}_r': observed_rvals})
        df.to_csv(base + "_subject_rvals.csv", index=False)

        # Save group-level observed correlation
        with open(base + f"_{stat}_observed_r.txt", "w") as f:
            f.write(f"{observed_value:.6f}")

        print(f"Saved results to {base}_*.npy/.txt/.csv")

    # Optional plot
    plt.figure(figsize=(8,6))
    plt.hist(null_distributions, bins=50, color='gray', alpha=0.7, density=True, label=f"Null Distribution ({null_type})")
    plt.axvline(observed_value, color='red', linestyle='dotted', linewidth=2, label=f"Observed {stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.xlabel(f"{stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.ylabel("Density")
    plt.title(f"Permutation Test ({null_type.replace('_', ' ').title()})")
    plt.grid(True)
    plt.legend()
    plt.show()

    return observed_value, null_distributions, p_value


In [ ]:
permutation_test_toSR(EEG_FF_predicted, FF_observed, EEG_Sher_predicted, Sher_observed, 
                     null_type="phase_randomization", stat="mean", 
                     n_permutations=10000, save_path='../4_results/2_classification/11_Movies_from_EEG_Rest/model5_p05_mean', corr_type="pearson")

permutation_test_toSR(pupil_FF_predicted, FF_observed, pupil_Sher_predicted, Sher_observed, 
                     null_type="phase_randomization", stat="mean", 
                     n_permutations=10000, save_path='../4_results/2_classification/13_Movies_from_Pupil_Rest/model5_p05_mean', corr_type="pearson")

permutation_test_toSR(paranoia_FF_predicted, FF_observed, paranoia_Sher_predicted, Sher_observed, 
                     null_type="phase_randomization", stat="mean", 
                     n_permutations=10000, save_path='../4_results/2_classification/15_Movies_from_Pupil_Story/model5_p05_mean', corr_type="pearson")


In [ ]:
FF_Sher_rvals = []
for i in range(FF_Sher_predicted.shape[0]):
    FF_Sher_rvals.append(pearsonr(FF_Sher_predicted[i], Sher_observed[i])[0])
FF_Sher_rvals = np.array(FF_Sher_rvals)
Sher_FF_rvals = []
for i in range(Sher_FF_predicted.shape[0]):
    Sher_FF_rvals.append(pearsonr(Sher_FF_predicted[i], FF_observed[i])[0])
Sher_FF_rvals = np.array(Sher_FF_rvals)

SR_rvals = np.concatenate((FF_Sher_rvals, Sher_FF_rvals))

SR_z_vals = fisher_z(SR_rvals)
SR_z_avg = np.mean(SR_z_vals)
SR_avg_r = fisher_z_inv(SR_z_avg)

count = 0
fake_r = []

for i in range(10000):
    shuffled_FF_observed = np.array([phase_randomize(a) for a in FF_observed])
    shuffled_Sher_observed = np.array([phase_randomize(a) for a in Sher_observed])

    fake_FF_Sher_rvals = []
    for j in range(FF_Sher_predicted.shape[0]):
        fake_FF_Sher_rvals.append(pearsonr(FF_Sher_predicted[j], shuffled_Sher_observed[j])[0])
    fake_FF_Sher_rvals = np.array(fake_FF_Sher_rvals)

    fake_Sher_FF_rvals = []
    for j in range(Sher_FF_predicted.shape[0]):
        fake_Sher_FF_rvals.append(pearsonr(Sher_FF_predicted[j], shuffled_FF_observed[j])[0])
    fake_Sher_FF_rvals = np.array(fake_Sher_FF_rvals)

    fake_all_rvals = np.concatenate((fake_FF_Sher_rvals, fake_Sher_FF_rvals))
    fake_z_vals = fisher_z(fake_all_rvals)
    fake_z_avg = np.mean(fake_z_vals)
    fake_avg_r = fisher_z_inv(fake_z_avg)
    fake_r.append(fake_avg_r)

    if fake_avg_r >= SR_avg_r:
        count += 1

p_value = (count + 1) / (10000 + 1)
print(f"SR from SR avg r: {SR_avg_r:.4f}, p-value: {p_value:.4f}")
fake_r = np.array(fake_r)

with open ('../4_results/2_classification/1_Movies_within/model5_p05_mean_pval.txt', "w") as f:
    f.write(f"{p_value:.6f}")
with open ('../4_results/2_classification/1_Movies_within/model5_p05_mean_mean_observed_r.txt', "w") as f:
    f.write(f"{SR_avg_r:.6f}")
np.save('../4_results/2_classification/1_Movies_within/model5_p05_mean_null_distribution.npy', fake_r)
np.save('../4_results/2_classification/1_Movies_within/model5_p05_mean_observed_rvals.npy', SR_rvals)